# 03 — ENTSO-E fundamentals

Downloads from the ENTSO-E Transparency Platform the **physical drivers** of the Italian price: load, wind and solar, both **forecast** and **actual**, for Italy as a whole. GME gives the effect (prices); here we take the causes.

- **Input:** ENTSO-E REST API (token in `.env`)
- **Output:** raw XML cached in `data/raw/entsoe/`, `data/processed/entsoe_hourly_2020_2025.parquet` (52,608 hours, mean MW per hour)
- **Key points:** from 31 Dec 2024 ENTSO-E publishes 15-minute values, averaged to the hour in UTC; gaps stay explicit as `NaN`. Forecasts become model features, actuals are diagnostics only.

In [19]:
# --- Setup: paths, imports, credentials -----------------------------------
from pathlib import Path
import os
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
RAW_ENTSOE = PROJECT_ROOT / "data" / "raw" / "entsoe"
PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW_ENTSOE.mkdir(parents=True, exist_ok=True)

ENTSOE_API_URL = "https://web-api.tp.entsoe.eu/api"
BIDDING_ZONE_NORTH = "10Y1001A1001A73I"   # EIC code of the Italian NORD bidding zone
ITALY = "10YIT-GRTN-----B"                # EIC code of the Italian country / control area

# ENTSO-E resolution codes -> minutes. PT60M is what Italy uses in our perimeter;
# the others are listed so that an unexpected resolution raises a KeyError
# instead of being silently mishandled.
RESOLUTION_TO_MINUTES = {"PT60M": 60, "PT30M": 30, "PT15M": 15}

# One day of margin on each side of every request window. Two reasons: the API
# drops the last MTU of the requested window, and the padding guarantees that
# consecutive monthly chunks overlap instead of leaving a hole at the seam. The
# overlap is removed later by drop_duplicates.
# Value recovered from the cached XML files, which record the window that was
# actually requested - do not change it, or the cache stops matching the code.
PAD = pd.Timedelta(days=1)

# load_dotenv reads the .env file and pushes its keys into os.environ. We point
# at the file explicitly: the notebook's working directory is notebooks/, so a
# bare load_dotenv() would look in the wrong folder.
load_dotenv(PROJECT_ROOT / ".env")
ENTSOE_TOKEN = os.getenv("ENTSOE_API_TOKEN")

# Fail loudly and early. A missing token would otherwise surface much later as
# a confusing 401 buried inside an XML error body.
assert ENTSOE_TOKEN, "ENTSOE_API_TOKEN not found - check the .env file"
print(f"Token loaded: {len(ENTSOE_TOKEN)} characters")   # never print the token itself

Token loaded: 36 characters


## Il downloader

Due scelte che vale la pena ricordare fra sei mesi:

- **si mette in cache l'XML grezzo, non il DataFrame.** La logica di parsing cambia
  mentre si impara il formato, la risposta HTTP no. Il suffisso di versione nel nome
  del file va incrementato quando cambia la *richiesta*, altrimenti la cache serve in
  silenzio dati ottenuti con parametri diversi;
- **si scarica per finestre mensili.** L'API rifiuta richieste più lunghe di un anno, e
  il margine di `PAD` per lato sforerebbe comunque il limite su una richiesta annuale.

Le finestre sono costruite in ora italiana e non in UTC: d'inverno il confine è alle
23:00Z, d'estate alle 22:00Z. Tagliare in UTC sfalserebbe i mesi di un'ora per metà anno.

In [20]:
# --- Generic ENTSO-E downloader with on-disk caching ----------------------
def entsoe_period(ts: pd.Timestamp) -> str:
    """ENTSO-E wants UTC timestamps as yyyyMMddHHmm."""
    return ts.tz_convert("UTC").strftime("%Y%m%d%H%M")


def fetch_entsoe(extra_params: dict, start: pd.Timestamp, end: pd.Timestamp,
                 cache_name: str, max_retries: int = 3) -> str:
    """Download one document and cache the raw XML next to the other raw data.

    Caching the RAW response (not the parsed frame) is deliberate: parsing
    logic changes as we learn the format, the network response does not.
    """
    cache_file = RAW_ENTSOE / f"{cache_name}.xml"
    if cache_file.exists():
        return cache_file.read_text()

    params = {
        "securityToken": ENTSOE_TOKEN,
        "periodStart": entsoe_period(start),
        "periodEnd": entsoe_period(end),
        **extra_params,
    }

    for attempt in range(max_retries):
        r = requests.get(ENTSOE_API_URL, params=params, timeout=60)
        if r.status_code == 200:
            cache_file.write_text(r.text)
            return r.text
        # 429 = rate limited, 5xx = their side. Both are worth retrying with a
        # growing pause; a 400 is our mistake and retrying would be useless.
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** attempt)
            continue
        # Do not truncate below ~1000 chars: the <Reason><text> element that
        # explains the 400 lives at the end of the error body.
        raise RuntimeError(f"HTTP {r.status_code} for {cache_name}: {r.text[:1000]}")

    raise RuntimeError(f"{cache_name}: giving up after {max_retries} retries")


def month_windows(year: int):
    """Yield (start, end) pairs covering one calendar year, month by month.

    Boundaries are Italian-time month starts, not UTC ones: an Italian month
    begins at 23:00Z in winter and at 22:00Z in summer.
    """
    months = pd.date_range(f"{year}-01-01", f"{year + 1}-01-01",
                           freq="MS", tz="Europe/Rome")
    # 13 boundaries -> 12 half-open [start, end) windows.
    for start, end in zip(months[:-1], months[1:]):
        yield start, end

## Il parser

Tre trappole del formato ENTSO-E, tutte silenziose se ignorate:

1. **Namespace.** Senza namespace `findall` restituisce lista vuota *senza errore*.
   Il numero di versione nella URN cambia nel tempo, quindi si legge dal tag della root.
2. **`curveType A03`.** Un `Point` viene pubblicato **solo quando il valore cambia**.
   Le posizioni mancanti ripetono la precedente, comprese quelle in coda — che non
   hanno un `max(position)` a rivelarle. Il numero atteso di posizioni si calcola dal
   `timeInterval` del `Period`. Sul solare consuntivo questo dimezza la serie: 374
   punti pubblicati per 744 ore, perché di notte lo zero si ripete.
3. **Più `TimeSeries` per documento.** Il carico ne ha una; A69 ne ha due (solare `B16`
   ed eolico onshore `B19`); A74 restituisce tutto il mix di generazione, con la stessa
   fonte spezzata in decine di `Period` (il pompaggio) e pubblicata in **due direzioni**:
   `generation` per l'energia immessa, `consumption` per quella assorbita — gli inverter
   fotovoltaici di notte, le pompe degli impianti di accumulo. Mescolarle metterebbe due
   valori diversi sullo stesso istante.

Il forward-fill si fa **dentro** ogni `Period` e mai fra un `Period` e il successivo:
nel buco fra due frammenti il dato non è "uguale a prima", è assente.

In [21]:
# --- Parse any ENTSO-E market document into long rows ---------------------
def parse_entsoe_document(xml_text: str, value_tag: str = "quantity") -> pd.DataFrame:
    """Parse an ENTSO-E market document into rows of
    (ts_utc, psr_type, direction, value).

    value_tag is "quantity" for load and generation documents, "price.amount"
    for A44 prices. psr_type is None for documents that are not split by
    production type. direction separates the two quantities that A74 publishes
    for the same psr_type: energy flowing INTO the zone (generation) and out of
    it (consumption).
    """
    root = ET.fromstring(xml_text)
    ns = {"ns": root.tag.split("}")[0].strip("{")}

    frames = []
    for time_series in root.findall(".//ns:TimeSeries", ns):
        psr = time_series.find(".//ns:MktPSRType/ns:psrType", ns)
        psr_type = psr.text if psr is not None else None
        direction = (
            "generation"
            if time_series.find("ns:inBiddingZone_Domain.mRID", ns) is not None
            else "consumption"
        )

        # One psr_type can be spread over several TimeSeries, and one TimeSeries
        # can hold several Periods. Each Period is rebuilt on its own and they
        # are concatenated afterwards: forward-filling ACROSS the gap between
        # two Periods would invent data where there is none.
        for period in time_series.findall("ns:Period", ns):
            start = pd.Timestamp(period.find("ns:timeInterval/ns:start", ns).text)
            end = pd.Timestamp(period.find("ns:timeInterval/ns:end", ns).text)
            step = RESOLUTION_TO_MINUTES[period.find("ns:resolution", ns).text]

            # Ask the document how many positions the period SHOULD have: with
            # curveType A03 the trailing points are omitted when the value stops
            # changing, so the last observed position is not the last position.
            n_positions = int((end - start) / pd.Timedelta(minutes=step))

            positions, values = [], []
            for point in period.findall("ns:Point", ns):
                value_node = point.find(f"ns:{value_tag}", ns)
                if value_node is None:
                    raise ValueError(
                        f"<{value_tag}> not found in a Point: wrong value tag "
                        f"for this document type?"
                    )
                positions.append(int(point.find("ns:position", ns).text))
                values.append(float(value_node.text))

            df = (
                pd.DataFrame({"position": positions, "value": values})
                .set_index("position")
                .reindex(range(1, n_positions + 1))
                .ffill()             # A03: a point exists only when the value changes
                .reset_index()
            )

            # A leading gap cannot be forward-filled - there is nothing before
            # it. If this fires, the document is malformed and we want to know.
            assert df["value"].notna().all(), (
                f"missing leading positions in period starting {start}"
            )

            df["ts_utc"] = start + pd.to_timedelta((df["position"] - 1) * step, unit="m")
            df["psr_type"] = psr_type
            df["direction"] = direction
            frames.append(df[["ts_utc", "psr_type", "direction", "value"]])

    return pd.concat(frames, ignore_index=True).sort_values("ts_utc").reset_index(drop=True)

In [22]:
# --- Download one year as padded monthly chunks, then trim to the year ----
def load_year_series(extra_params: dict, year: int, cache_prefix: str,
                     value_tag: str = "quantity",
                     cache_version: str = "v3") -> pd.DataFrame:
    year_start = pd.Timestamp(f"{year}-01-01", tz="Europe/Rome")
    year_end = pd.Timestamp(f"{year + 1}-01-01", tz="Europe/Rome")

    chunks = []
    for start, end in month_windows(year):
        xml_text = fetch_entsoe(
            extra_params,
            start - PAD,
            end + PAD,
            cache_name=f"{cache_prefix}_{start:%Y%m}_{cache_version}",
        )
        chunks.append(parse_entsoe_document(xml_text, value_tag=value_tag))
        time.sleep(0.2)   # the limit is 400 req/min; we stay well under it

    df = pd.concat(chunks, ignore_index=True)

    # Deduplicate on the FULL key, not on ts_utc alone. The padding makes
    # consecutive chunks overlap by a day, which is what we are removing here -
    # but a single timestamp now carries one row per (psr_type, direction), and
    # dropping on ts_utc alone would keep solar and throw away wind.
    df = df.drop_duplicates(subset=["ts_utc", "psr_type", "direction"]).sort_values("ts_utc")

    df["ts_local"] = df["ts_utc"].dt.tz_convert("Europe/Rome")
    mask = (df["ts_local"] >= year_start) & (df["ts_local"] < year_end)
    return df.loc[mask].reset_index(drop=True)

## Validazione incrociata GME ↔ ENTSO-E

Due fonti indipendenti per lo stesso oggetto: il prezzo orario della zona NORD nel 2021.
Il GME lo pubblica in XLSX con `Data` e `Ora` 1–25, ENTSO-E in XML con posizioni e
`timeInterval`. Se le due ricostruzioni coincidono ora per ora, allora sono corrette
entrambe la costruzione del timestamp di `01_ingestion` (giorni da 23 e da 25 ore
compresi) e la logica delle posizioni di questo notebook.

Il join è su `ts_utc`: è l'unica rappresentazione su cui due fonti non possono divergere
per convenzione. `how="outer"` con `indicator=True` fa emergere le ore presenti in una
sola delle due.

In [23]:
# --- ENTSO-E side: day-ahead prices for the NORD zone, 2021 ---------------
prices_2021 = load_year_series(
    {"documentType": "A44",
     "in_Domain": BIDDING_ZONE_NORTH,
     "out_Domain": BIDDING_ZONE_NORTH},
    2021,
    cache_prefix="price_nord",
    value_tag="price.amount",
)

# 24 is the norm; 23 = spring DST switch, 25 = autumn one. Anything else is a bug.
hours_per_day = prices_2021.groupby(prices_2021["ts_local"].dt.date).size()
print(prices_2021.shape)
print(prices_2021["ts_local"].min(), "->", prices_2021["ts_local"].max())
print(hours_per_day[hours_per_day != 24])

(8760, 5)
2021-01-01 00:00:00+01:00 -> 2021-12-31 23:00:00+01:00
ts_local
2021-03-28    23
2021-10-31    25
dtype: int64


In [24]:
# --- Cross-validation: GME (XLSX) vs ENTSO-E (API), same hours, same zone --
gme = pd.read_parquet(PROCESSED / "mgp_prices_hourly_2021_2025q3.parquet")

gme_nord_2021 = (
    gme.loc[gme["ts_local"].dt.year == 2021, ["ts_utc", "NORD"]]
       .rename(columns={"NORD": "price_gme"})
)

merged = gme_nord_2021.merge(
    prices_2021[["ts_utc", "value"]].rename(columns={"value": "price_entsoe"}),
    on="ts_utc",
    how="outer",        # outer, so that hours present in only ONE source show up
    indicator=True,
)

print("rows:", len(merged))
print(merged["_merge"].value_counts())      # both / left_only / right_only

both = merged[merged["_merge"] == "both"].copy()
both["diff"] = (both["price_gme"] - both["price_entsoe"]).abs()

# GME publishes 3+ decimals, ENTSO-E rounds to the cent: 0.005 is the largest
# difference that pure rounding can produce. Anything above that is a real
# disagreement between the two sources.
print("max abs diff :", both["diff"].max())
print("hours > 0.01 :", (both["diff"] > 0.01).sum())

rows: 8760
_merge
both          8760
left_only        0
right_only       0
Name: count, dtype: int64
max abs diff : 0.0049999999999954525
hours > 0.01 : 0


## Le quattro serie da scaricare

| Serie | documentType | processType | Dominio | Ruolo |
|---|---|---|---|---|
| Carico previsto day-ahead | A65 | A01 | `outBiddingZone_Domain` | **Feature** |
| Carico consuntivo | A65 | A16 | `outBiddingZone_Domain` | Diagnostica |
| Eolico e solare previsti | A69 | A01 | `in_Domain` | **Feature** |
| Generazione consuntiva | A74 | A16 | `in_Domain` | Diagnostica |

Il nome del parametro di dominio cambia col tipo di documento: il carico è un consumo
(`outBiddingZone_Domain`), la generazione avviene dentro la zona (`in_Domain`).
Sbagliarlo dà HTTP 400, che almeno fallisce rumorosamente.

**Le serie `_actual` non entrano mai nel modello.** Alle 12:00 del giorno prima, quando
si formulano le offerte sul MGP, non esistono. Servono a misurare l'errore di previsione
del TSO — che è a sua volta un driver di volatilità del prezzo — e come benchmark.

`residual load` = carico previsto − eolico previsto − solare previsto. È la domanda che
gli impianti programmabili devono coprire, e quindi ciò che determina fin dove si sale
nella curva di offerta.

In [25]:
# --- Probe the four series: one month each, before downloading five years --
SERIES_SPECS = {
    "load_forecast":       {"documentType": "A65", "processType": "A01", "outBiddingZone_Domain": ITALY},
    "load_actual":         {"documentType": "A65", "processType": "A16", "outBiddingZone_Domain": ITALY},
    "wind_solar_forecast": {"documentType": "A69", "processType": "A01", "in_Domain": ITALY},
    "generation_actual":   {"documentType": "A74", "processType": "A16", "in_Domain": ITALY},
}

probe_start = pd.Timestamp("2024-01-01", tz="Europe/Rome")
probe_end = pd.Timestamp("2024-02-01", tz="Europe/Rome")

for name, spec in SERIES_SPECS.items():
    print("=" * 72)
    print(name, "->", spec)
    try:
        xml_text = fetch_entsoe(spec, probe_start, probe_end, cache_name=f"probe_{name}_202401")
    except RuntimeError as err:
        print("FAILED:", err)          # a 400 means the parameter names are wrong
        continue

    root = ET.fromstring(xml_text)
    ns = {"ns": root.tag.split("}")[0].strip("{")}

    # An Acknowledgement_MarketDocument means "request accepted, no data": that
    # is the answer to "does this series exist for this domain?".
    print("root:", root.tag.split("}")[-1])

    series = root.findall(".//ns:TimeSeries", ns)
    print("TimeSeries in document:", len(series))

    for ts in series[:4]:
        psr = ts.find(".//ns:MktPSRType/ns:psrType", ns)
        period = ts.find("ns:Period", ns)
        first_point = period.find("ns:Point", ns)
        # Which child tags does a Point carry? This is what tells us whether the
        # value lives in <quantity> or in <price.amount>.
        point_tags = [child.tag.split("}")[-1] for child in first_point]
        print(
            f"  psrType={psr.text if psr is not None else None}"
            f" | resolution={period.find('ns:resolution', ns).text}"
            f" | start={period.find('ns:timeInterval/ns:start', ns).text}"
            f" | point tags={point_tags}"
        )

load_forecast -> {'documentType': 'A65', 'processType': 'A01', 'outBiddingZone_Domain': '10YIT-GRTN-----B'}
root: GL_MarketDocument
TimeSeries in document: 1
  psrType=None | resolution=PT60M | start=2023-12-31T23:00Z | point tags=['position', 'quantity']
load_actual -> {'documentType': 'A65', 'processType': 'A16', 'outBiddingZone_Domain': '10YIT-GRTN-----B'}
root: GL_MarketDocument
TimeSeries in document: 1
  psrType=None | resolution=PT60M | start=2023-12-31T23:00Z | point tags=['position', 'quantity']
wind_solar_forecast -> {'documentType': 'A69', 'processType': 'A01', 'in_Domain': '10YIT-GRTN-----B'}
root: GL_MarketDocument
TimeSeries in document: 2
  psrType=B16 | resolution=PT60M | start=2023-12-31T23:00Z | point tags=['position', 'quantity']
  psrType=B19 | resolution=PT60M | start=2023-12-31T23:00Z | point tags=['position', 'quantity']
generation_actual -> {'documentType': 'A74', 'processType': 'A16', 'in_Domain': '10YIT-GRTN-----B'}
root: GL_MarketDocument
TimeSeries in docume

In [26]:
# --- Full inventory of the A74 document: it is not just wind and solar -----
PSR_TYPE_NAMES = {
    "B01": "Biomass", "B02": "Fossil brown coal", "B03": "Fossil coal-derived gas",
    "B04": "Fossil gas", "B05": "Fossil hard coal", "B06": "Fossil oil",
    "B09": "Geothermal", "B10": "Hydro pumped storage", "B11": "Hydro run-of-river",
    "B12": "Hydro water reservoir", "B14": "Nuclear", "B15": "Other renewable",
    "B16": "Solar", "B17": "Waste", "B18": "Wind offshore", "B19": "Wind onshore",
    "B20": "Other",
}

xml_text = fetch_entsoe(
    SERIES_SPECS["generation_actual"], probe_start, probe_end,
    cache_name="probe_generation_actual_202401",
)
root = ET.fromstring(xml_text)
ns = {"ns": root.tag.split("}")[0].strip("{")}

rows = []
for time_series in root.findall(".//ns:TimeSeries", ns):
    psr = time_series.find(".//ns:MktPSRType/ns:psrType", ns)
    psr_code = psr.text if psr is not None else None
    direction = (
        "generation"
        if time_series.find("ns:inBiddingZone_Domain.mRID", ns) is not None
        else "consumption"
    )
    for period in time_series.findall("ns:Period", ns):
        rows.append({
            "psr_type": psr_code,
            "name": PSR_TYPE_NAMES.get(psr_code, "?"),
            "direction": direction,
            "start": period.find("ns:timeInterval/ns:start", ns).text,
            "end": period.find("ns:timeInterval/ns:end", ns).text,
            "n_points": len(period.findall("ns:Point", ns)),
        })

inventory = pd.DataFrame(rows).sort_values(["psr_type", "direction", "start"])
print(inventory.to_string(index=False))
print("\nnumber of periods per psrType:")
print(inventory["psr_type"].value_counts().to_string())

psr_type                    name   direction             start               end  n_points
     B01                 Biomass  generation 2023-12-31T23:00Z 2024-01-31T23:00Z       662
     B03 Fossil coal-derived gas  generation 2023-12-31T23:00Z 2024-01-31T23:00Z       665
     B04              Fossil gas  generation 2023-12-31T23:00Z 2024-01-31T23:00Z       742
     B05        Fossil hard coal  generation 2023-12-31T23:00Z 2024-01-31T23:00Z       632
     B06              Fossil oil  generation 2023-12-31T23:00Z 2024-01-31T23:00Z       332
     B09              Geothermal  generation 2023-12-31T23:00Z 2024-01-31T23:00Z       442
     B10    Hydro pumped storage consumption 2023-12-31T23:00Z 2024-01-01T16:00Z        12
     B10    Hydro pumped storage consumption 2024-01-01T21:00Z 2024-01-05T05:00Z        52
     B10    Hydro pumped storage consumption 2024-01-05T22:00Z 2024-01-05T23:00Z         1
     B10    Hydro pumped storage consumption 2024-01-06T01:00Z 2024-01-06T06:00Z         2

## Download completo

288 richieste: 4 serie × 6 anni × 12 mesi. La prima esecuzione richiede qualche minuto,
le successive leggono dalla cache.

Il 2020 viene scaricato intero anche se serve solo dicembre: tiene il loader semplice, e
il taglio al perimetro (1 gennaio 2021 → 30 settembre 2025) si fa in `05_dataset`. I mesi
precedenti servono comunque alle feature ritardate sulle prime ore del 2021.

In [27]:
# --- Full download: four series, six calendar years -----------------------
YEARS = range(2020, 2026)

DOWNLOAD_PLAN = {
    "load_forecast":       {"spec": SERIES_SPECS["load_forecast"],       "prefix": "load_fc"},
    "load_actual":         {"spec": SERIES_SPECS["load_actual"],         "prefix": "load_act"},
    "wind_solar_forecast": {"spec": SERIES_SPECS["wind_solar_forecast"], "prefix": "wind_solar_fc"},
    "generation_actual":   {"spec": SERIES_SPECS["generation_actual"],   "prefix": "gen_act"},
}

raw_series = {}
for name, cfg in DOWNLOAD_PLAN.items():
    frames = []
    for year in YEARS:
        print(f"{name} {year} ...", end=" ", flush=True)
        frames.append(load_year_series(cfg["spec"], year, cache_prefix=cfg["prefix"]))
        print("ok")
    raw_series[name] = pd.concat(frames, ignore_index=True)
    print(f"-> {name}: {len(raw_series[name])} rows\n")

load_forecast 2020 ... ok
load_forecast 2021 ... ok
load_forecast 2022 ... ok
load_forecast 2023 ... ok
load_forecast 2024 ... ok
load_forecast 2025 ... ok
-> load_forecast: 78884 rows

load_actual 2020 ... ok
load_actual 2021 ... ok
load_actual 2022 ... ok
load_actual 2023 ... ok
load_actual 2024 ... ok
load_actual 2025 ... ok
-> load_actual: 78957 rows

wind_solar_forecast 2020 ... ok
wind_solar_forecast 2021 ... ok
wind_solar_forecast 2022 ... ok
wind_solar_forecast 2023 ... ok
wind_solar_forecast 2024 ... ok
wind_solar_forecast 2025 ... ok
-> wind_solar_forecast: 147924 rows

generation_actual 2020 ... ok
generation_actual 2021 ... ok
generation_actual 2022 ... ok
generation_actual 2023 ... ok
generation_actual 2024 ... ok
generation_actual 2025 ... ok
-> generation_actual: 1169911 rows



In [28]:
# --- What did we actually get, over the whole period? ---------------------
# Which sources appear, and from when? Offshore wind cannot exist before 2022,
# and a column that is empty for half the series must be treated as such rather
# than filled with zeros for convenience.
for name, df in raw_series.items():
    print("=" * 72)
    print(f"{name}: {len(df)} rows, {df['ts_utc'].min()} -> {df['ts_utc'].max()}")
    print(
        df.groupby(["psr_type", "direction"])["ts_utc"]
          .agg(["count", "min", "max"])
          .to_string()
    )

load_forecast: 78884 rows, 2019-12-31 23:00:00+00:00 -> 2025-12-31 22:45:00+00:00
Empty DataFrame
Columns: [count, min, max]
Index: []
load_actual: 78957 rows, 2019-12-31 23:00:00+00:00 -> 2025-12-31 22:45:00+00:00
Empty DataFrame
Columns: [count, min, max]
Index: []
wind_solar_forecast: 147924 rows, 2019-12-31 23:00:00+00:00 -> 2025-12-31 22:45:00+00:00
                     count                       min                       max
psr_type direction                                                            
B16      generation  77778 2019-12-31 23:00:00+00:00 2025-12-31 22:45:00+00:00
B19      generation  70146 2020-12-14 23:00:00+00:00 2025-12-31 22:45:00+00:00
generation_actual: 1169911 rows, 2019-12-31 23:00:00+00:00 -> 2025-12-31 22:45:00+00:00
                      count                       min                       max
psr_type direction                                                             
B01      generation   78957 2019-12-31 23:00:00+00:00 2025-12-31 22:45:00+00:00

In [29]:
# --- Diagnostic: actual time step of each series ---------------------------
# The parser accepts both PT60M and PT15M, so a change of resolution goes
# through silently. Here we measure the real step between consecutive
# timestamps, separately for each (psr_type, direction).
for name, df in raw_series.items():
    df = df.sort_values("ts_utc")

    # dropna=False: load documents have psr_type = None, and groupby drops
    # None keys by default (that is why cell 15 printed an empty table)
    step_min = (
        df.groupby(["psr_type", "direction"], dropna=False)["ts_utc"]
          .diff()
          .dt.total_seconds() / 60
    )

    steps = pd.DataFrame({"ts_local": df["ts_local"], "step_min": step_min}).dropna()

    # Steps up to 60 min are the regular grid; anything longer is a hole
    # between two Periods (typical for pumped storage, which is fragmented)
    regular = steps[steps["step_min"] <= 60]

    print("=" * 72)
    print(name)
    print(regular.groupby("step_min")["ts_local"].agg(["count", "min", "max"]).to_string())
    print("holes (step > 60 min):", (steps["step_min"] > 60).sum())

load_forecast
          count                       min                       max
step_min                                                           
15.0      35035 2025-01-01 01:15:00+01:00 2025-12-31 23:45:00+01:00
60.0      43847 2020-01-01 01:00:00+01:00 2024-12-31 23:00:00+01:00
holes (step > 60 min): 1
load_actual
          count                       min                       max
step_min                                                           
15.0      35131 2024-12-31 01:15:00+01:00 2025-12-31 23:45:00+01:00
60.0      43825 2020-01-01 01:00:00+01:00 2024-12-31 01:00:00+01:00
holes (step > 60 min): 0
wind_solar_forecast
          count                       min                       max
step_min                                                           
15.0      69876 2025-01-01 01:15:00+01:00 2025-12-31 23:45:00+01:00
60.0      77991 2020-01-01 01:00:00+01:00 2024-12-31 23:00:00+01:00
holes (step > 60 min): 55
generation_actual
           count                       min  

In [30]:
# --- Harmonise every series to hourly resolution ---------------------------
# Only generation documents need the direction filter. Load documents use
# outBiddingZone_Domain, so the parser labels ALL their rows "consumption":
# filtering them on "generation" would delete the whole load series.
GENERATION_SERIES = {"wind_solar_forecast", "generation_actual"}


def to_hourly(df: pd.DataFrame, keep_generation_only: bool) -> pd.DataFrame:
    """Aggregate a long ENTSO-E frame to one row per (hour, psr_type).

    Values are MW, i.e. AVERAGE power over the MTU: the hourly value is the
    MEAN of the quarter-hours (summing would give 4x the real figure).
    n_obs counts how many raw points fell into each hour, so that incomplete
    hours are flagged instead of being averaged silently.
    """
    out = df.copy()
    if keep_generation_only:
        out = out[out["direction"] == "generation"]

    out["minute"] = out["ts_utc"].dt.minute
    # Floor in UTC, never in local time: UTC has no DST, so every hour
    # contains exactly the quarter-hours that belong to it
    out["ts_utc"] = out["ts_utc"].dt.floor("h")

    hourly = (
        out.groupby(["ts_utc", "psr_type"], dropna=False)   # dropna=False: load has psr_type None
           .agg(value=("value", "mean"),
                n_obs=("value", "size"),
                max_minute=("minute", "max"))
           .reset_index()
    )

    # If any point in the hour sits at :15/:30/:45 the hour is quarter-hourly
    # and needs 4 points; otherwise it is an hourly point and needs 1
    hourly["expected_obs"] = (hourly["max_minute"] > 0).map({True: 4, False: 1})
    hourly["incomplete"] = hourly["n_obs"] != hourly["expected_obs"]
    return hourly.drop(columns=["max_minute"])


hourly_series = {}
for name, df in raw_series.items():
    hourly = to_hourly(df, keep_generation_only=name in GENERATION_SERIES)
    hourly_series[name] = hourly

    incomplete = hourly[hourly["incomplete"]]
    print("=" * 72)
    print(f"{name}: {len(df)} raw rows -> {len(hourly)} hourly rows")
    print("incomplete hours:", len(incomplete))
    if len(incomplete):
        print(incomplete.groupby("psr_type", dropna=False)["n_obs"]
                        .agg(["count", "min", "max"]).to_string())

load_forecast: 78884 raw rows -> 52607 hourly rows
incomplete hours: 0
load_actual: 78957 raw rows -> 52608 hourly rows
incomplete hours: 0
wind_solar_forecast: 147924 raw rows -> 95514 hourly rows
incomplete hours: 0
generation_actual: 1169911 raw rows -> 714668 hourly rows
incomplete hours: 2745
          count  min  max
psr_type                 
B09           5    2    3
B10         506    1    3
B15        1258    1    3
B19           5    1    2
B25         971    1    3


In [31]:
# --- Completeness check on the full hourly grid ----------------------------
# Every hour from 1 Jan 2020 to 31 Dec 2025, Italian calendar, expressed in UTC.
# date_range on a tz-aware start counts real elapsed hours, so DST days get
# 23 and 25 hours automatically.
FULL_GRID = pd.date_range(
    pd.Timestamp("2020-01-01", tz="Europe/Rome"),
    pd.Timestamp("2026-01-01", tz="Europe/Rome"),
    freq="h",
    inclusive="left",
).tz_convert("UTC")
assert len(FULL_GRID) == 52608, len(FULL_GRID)   # 2 leap years + 4 normal ones

# Model perimeter: only gaps inside it really matter
PERIMETER_START = pd.Timestamp("2021-01-01", tz="Europe/Rome")
PERIMETER_END = pd.Timestamp("2025-10-01", tz="Europe/Rome")

# Candidate columns: forecasts are the features, actuals are only diagnostics
COLUMNS_TO_CHECK = {
    "load_forecast":       [None],
    "load_actual":         [None],
    "wind_solar_forecast": ["B16", "B19"],
    "generation_actual":   ["B16", "B19", "B18", "B04"],
}


def find_gaps(hourly: pd.DataFrame, psr_code) -> pd.DataFrame:
    """Return one row per contiguous block of missing hours.

    Incomplete hours (fewer quarter-hours than expected) count as missing:
    a mean over 1 point out of 4 is not a reliable hourly value.
    """
    # psr_type None becomes NaN after groupby, so it needs isna() instead of ==
    if psr_code is None:
        sel = hourly["psr_type"].isna()
    else:
        sel = hourly["psr_type"] == psr_code
    present = pd.DatetimeIndex(hourly.loc[sel & ~hourly["incomplete"], "ts_utc"])

    missing = pd.Series(FULL_GRID.difference(present))
    if missing.empty:
        return pd.DataFrame(columns=["start_local", "end_local", "n_hours"])

    # A new gap starts wherever the distance from the previous missing hour
    # is not exactly one hour
    gap_id = (missing.diff() != pd.Timedelta(hours=1)).cumsum()
    gaps = missing.groupby(gap_id).agg(["min", "max", "size"])
    gaps.columns = ["start_local", "end_local", "n_hours"]
    gaps["start_local"] = gaps["start_local"].dt.tz_convert("Europe/Rome")
    gaps["end_local"] = gaps["end_local"].dt.tz_convert("Europe/Rome")
    return gaps.reset_index(drop=True)


for name, codes in COLUMNS_TO_CHECK.items():
    for code in codes:
        gaps = find_gaps(hourly_series[name], code)
        in_perimeter = gaps[(gaps["end_local"] >= PERIMETER_START) &
                            (gaps["start_local"] < PERIMETER_END)]
        print("=" * 72)
        print(f"{name} | {code} ({PSR_TYPE_NAMES.get(code, 'load')})")
        print(f"missing hours: {gaps['n_hours'].sum()} in {len(gaps)} gaps "
              f"| inside perimeter: {in_perimeter['n_hours'].sum()} in {len(in_perimeter)} gaps")
        if len(in_perimeter):
            # The longest gaps are the ones that decide how to fill (or not)
            print(in_perimeter.sort_values("n_hours", ascending=False).head(5).to_string(index=False))

load_forecast | None (load)
missing hours: 1 in 1 gaps | inside perimeter: 1 in 1 gaps
              start_local                 end_local  n_hours
2025-01-01 00:00:00+01:00 2025-01-01 00:00:00+01:00        1
load_actual | None (load)
missing hours: 0 in 0 gaps | inside perimeter: 0 in 0 gaps
wind_solar_forecast | B16 (Solar)
missing hours: 1035 in 42 gaps | inside perimeter: 243 in 11 gaps
              start_local                 end_local  n_hours
2023-10-29 00:00:00+02:00 2023-10-29 23:00:00+01:00       25
2024-10-27 00:00:00+02:00 2024-10-27 23:00:00+01:00       25
2022-09-19 00:00:00+02:00 2022-09-19 23:00:00+02:00       24
2023-07-04 00:00:00+02:00 2023-07-04 23:00:00+02:00       24
2023-07-15 00:00:00+02:00 2023-07-15 23:00:00+02:00       24
wind_solar_forecast | B19 (Wind onshore)
missing hours: 8667 in 14 gaps | inside perimeter: 267 in 12 gaps
              start_local                 end_local  n_hours
2023-10-29 00:00:00+02:00 2023-10-29 23:00:00+01:00       25
2024-10-27 

In [32]:
# --- Pivot to wide format: one row per hour, one column per series ---------
# (series name in raw_series, psr_type) -> output column. Units stay MW.
COLUMN_MAP = {
    # features: published before the MGP gate closure
    ("load_forecast", None):         "load_forecast_mw",
    ("wind_solar_forecast", "B16"):  "solar_forecast_mw",
    ("wind_solar_forecast", "B19"):  "wind_onshore_forecast_mw",
    # diagnostics only: NEVER used as model inputs (not known at 12:00 D-1)
    ("load_actual", None):           "load_actual_mw",
    ("generation_actual", "B16"):    "solar_actual_mw",
    ("generation_actual", "B19"):    "wind_onshore_actual_mw",
    ("generation_actual", "B18"):    "wind_offshore_actual_mw",
    ("generation_actual", "B04"):    "gas_actual_mw",
}

columns = {}
for (name, code), col in COLUMN_MAP.items():
    hourly = hourly_series[name]
    sel = hourly["psr_type"].isna() if code is None else hourly["psr_type"] == code
    series = hourly.loc[sel].set_index("ts_utc")

    # Incomplete hours become NaN: a mean over 1-3 quarter-hours out of 4
    # would look like a real value and nobody would notice later
    columns[col] = series["value"].mask(series["incomplete"])

# reindex on the full grid: missing hours become explicit NaN rows instead of
# silently absent rows. NO filling here - imputation is a modelling choice and
# belongs to 05_dataset, where it can be flagged.
entsoe_hourly = pd.DataFrame(columns).reindex(FULL_GRID)
entsoe_hourly.index.name = "ts_utc"
entsoe_hourly = entsoe_hourly.reset_index()
entsoe_hourly.insert(1, "ts_local", entsoe_hourly["ts_utc"].dt.tz_convert("Europe/Rome"))

# --- Sanity checks ---------------------------------------------------------
assert len(entsoe_hourly) == len(FULL_GRID)
assert entsoe_hourly["ts_utc"].is_unique

# reindex must not have thrown values away: if the timestamps did not match
# the grid exactly (wrong tz, wrong unit), valid values would turn into NaN
for col, series in columns.items():
    assert series.notna().sum() == entsoe_hourly[col].notna().sum(), col

print(entsoe_hourly.shape)
print(entsoe_hourly.isna().sum().to_string())

(52608, 10)
ts_utc                          0
ts_local                        0
load_forecast_mw                1
solar_forecast_mw            1035
wind_onshore_forecast_mw     8667
load_actual_mw                  0
solar_actual_mw                 0
wind_onshore_actual_mw         13
wind_offshore_actual_mw     21047
gas_actual_mw                   0


In [33]:
# --- Persist and verify the round-trip -------------------------------------
# Full 2020-2025 range on purpose: the perimeter cut happens in 05_dataset,
# and December 2020 is needed for lagged features on the first hours of 2021
out_path = PROCESSED / "entsoe_hourly_2020_2025.parquet"
entsoe_hourly.to_parquet(out_path, index=False)

# Never trust a write you haven't read back: assert_frame_equal checks values,
# NaN positions AND dtypes (tz-aware timestamps included)
check = pd.read_parquet(out_path)
pd.testing.assert_frame_equal(check, entsoe_hourly)

print(f"saved {len(check)} rows, {out_path.stat().st_size / 1e6:.1f} MB -> {out_path.name}")
print(check.dtypes.to_string())

saved 52608 rows, 2.1 MB -> entsoe_hourly_2020_2025.parquet
ts_utc                              datetime64[us, UTC]
ts_local                    datetime64[us, Europe/Rome]
load_forecast_mw                                float64
solar_forecast_mw                               float64
wind_onshore_forecast_mw                        float64
load_actual_mw                                  float64
solar_actual_mw                                 float64
wind_onshore_actual_mw                          float64
wind_offshore_actual_mw                         float64
gas_actual_mw                                   float64
